In [ ]:
from src.preprocessing.preprocessing import processor
from src.models.lstm import LSTMModel
from src.utils.helpers import create_sequences
from src.postprocessing.inference import FloodClassifier

SITE_ID = "06820500"


In [ ]:
STATIC_FEATURES = [
    "longitude", "latitude", "DRAIN_SQKM", "artificial_path_pct",
    "wb5100_ann_mm", "snw_pc_syr", "snow_ice_nlcd06", "barren_nlcd06",
    "mains100_plant", "hga", "hgc", "bulk_density_avg", "elev_max_m", "aspect_deg"
]

DYNAMIC_FEATURES = [
    "streamflow_cfs_mean", "streamflow_cfs_max", "streamflow_cfs_min",
    "gage_height_ft_mean",
    "precipitation_mm",
    "temperature_c",
    "potential_evaporation_mm",
    "specific_humidity_kgkg",
    "shortwave_radiation_wm2",
    "longwave_radiation_wm2",
    "wind_speed_ms",
    "surface_pressure_pa",
    "cape_jkg",
    "convective_precip_fraction",
]

WINDOW_SIZE = 72

config = {
    "input_cols": DYNAMIC_FEATURES + STATIC_FEATURES,
    "static_cols": STATIC_FEATURES,
    "target": "streamflow_cfs_target_24h",
    "train_split": 0.8,
    "val_split": 0.9,
    "file_path": "flood-dataset-top30:v0",
    "file_name": "flood_model_top30",
    "table": "wandb.flood_model_top30",
    "lag_window": 1,
    "frequency": "hourly",
    "split_time_days": 30,
    "site_scaling": False,
    "sites": [SITE_ID],
}
pcr = processor(config)
pcr.pull_wandb()


wandb: Downloading large artifact 'flood-dataset-top30:v0', 145.19MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.2 (930.7MB/s)


Starting preprocessing: 141,711 rows, 31 columns
After lag null removal: 135,797 rows, 28 features
Splitting by time...
Train: 108,637 | Val: 13,580 | Test: 13,580
Scaling features and targets...
Scaling complete.


In [61]:
classifier = FloodClassifier()
_tmp = LSTMModel()
asymmetric_mse = _tmp._loss()

model = LSTMModel.load_model(
    "lstm_model_2xloss.keras",
    custom_objects={"asymmetric_mse": asymmetric_mse}
)

In [66]:
inference_df = pcr.prep_inference(wandb_config={
    "file_path": "flood-dataset-top30",
    "file_name": "flood_model_top30",
    "start_date": "2021-6-22",                 
    "end_date":   "2021-6-28",
    "frequency":  "hourly",                     
})

wandb: Downloading large artifact 'flood-dataset-top30:latest', 145.19MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.2 (844.1MB/s)


In [67]:
import numpy as np
import polars as pl
from datetime import timedelta
pl.Config.set_tbl_rows(50)
X_data, _, train_site_ids = create_sequences(inference_df, None, 72)

mask = np.array(train_site_ids) == "06820500"
X_site = X_data[mask]
site_ids = np.array(train_site_ids)[mask]

preds = model.predict(X_site)
unscaled_preds = pcr.unscale(preds, site_ids)
results = classifier.classify(unscaled_preds, site_ids)

# forecast_at = end of the 72h window + 24h ahead
timestamps = [
    t + timedelta(hours=24)
    for t in inference_df.filter(pl.col("site_id") == "06820500")["observation_hour"][72:].to_list()
]
df = pl.DataFrame({
    "forecast_at": timestamps,
    "predicted_cfs": unscaled_preds.tolist(),
    **{rp: [r[rp] for r in results] for rp in results[0]},
})
df


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step


forecast_at,predicted_cfs,Q2_cfs,Q5_cfs,Q10_cfs,Q25_cfs,Q50_cfs,Q100_cfs
datetime[μs],list[f64],bool,bool,bool,bool,bool,bool
2021-06-26 00:00:00,[1125.425515],false,false,false,false,false,false
2021-06-26 01:00:00,[1176.298194],false,false,false,false,false,false
2021-06-26 02:00:00,[1274.88292],false,false,false,false,false,false
2021-06-26 03:00:00,[1345.425935],false,false,false,false,false,false
2021-06-26 04:00:00,[1535.139608],false,false,false,false,false,false
2021-06-26 05:00:00,[2019.172799],false,false,false,false,false,false
2021-06-26 06:00:00,[3172.54456],false,false,false,false,false,false
2021-06-26 07:00:00,[4283.228302],false,false,false,false,false,false
2021-06-26 08:00:00,[5643.611006],false,false,false,false,false,false
